In [3]:
import pandas as pd
df = pd.read_csv('/Users/krishnagawande/agro/tempdata/weather.csv')

# Drop rows with duplicate values in the 'city' column
# keep='first' keeps the first occurrence and drops subsequent duplicates
df_unique_cities = df.drop_duplicates(subset=['city'], keep=False)

# Optionally, save the cleaned data back to a new CSV file
df_unique_cities.to_csv('/Users/krishnagawande/agro/tempdata/weather_unique_cities.csv', index=False)


In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

# 1. Load the dataset
df = pd.read_csv('/Users/krishnagawande/agro/tempdata/weather_unique_cities.csv')

# Drop any rows with missing coordinates just in case
df = df.dropna(subset=['lat', 'lng'])

# 2. Extract coordinates for clustering
coords = df[['lat', 'lng']].values

# 3. Define the number of clusters (adjust 'n_clusters' based on how many cities you want to keep)
# For example, if you want roughly 1000 representative cities across India:
N_CLUSTERS = 1000

print(f"Clustering {len(df)} cities into {N_CLUSTERS} geographic regions...")

# 4. Apply K-Means Clustering
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(coords)

# 5. Find the city closest to the center of each cluster
cluster_centers = kmeans.cluster_centers_
representative_indices = []

for i in range(N_CLUSTERS):
    # Get all cities in the current cluster
    cluster_points = df[df['cluster'] == i][['lat', 'lng']].values
    
    # Calculate the distance from each city in the cluster to the cluster center
    center = cluster_centers[i].reshape(1, -1)
    distances = cdist(cluster_points, center, metric='euclidean')
    
    # Get the index of the city with the minimum distance to the center
    # Note: We need to map this back to the original dataframe index
    cluster_indices = df[df['cluster'] == i].index
    closest_city_idx = cluster_indices[np.argmin(distances)]
    
    representative_indices.append(closest_city_idx)

# 6. Create the final dataframe with only the representative cities
df_clustered = df.loc[representative_indices].drop(columns=['cluster'])

# 7. Save the clustered data
output_file = '/Users/krishnagawande/agro/tempdata/weather_clustered_cities.csv'
df_clustered.to_csv(output_file, index=False)

print(f"Original row count: {len(df)}")
print(f"Row count after geospatial clustering: {len(df_clustered)}")
print(f"Saved representative cities to: {output_file}")


Clustering 5333 cities into 1000 geographic regions...
Original row count: 5333
Row count after geospatial clustering: 1000
Saved representative cities to: /Users/krishnagawande/agro/tempdata/weather_clustered_cities.csv


In [7]:
import pandas as pd

df = pd.read_csv('/Users/krishnagawande/agro/tempdata/weather_clustered_cities.csv')

# Count cities per state
state_counts = (
    df.groupby('state')['city']
      .count()
      .sort_values(ascending=False)
)

print(state_counts)


state
Andhra Pradesh       152
Tamil Nadu           105
Madhya Pradesh       105
Bihar                 98
Karnataka             98
West Bengal           51
Uttar Pradesh         50
Maharashtra           46
Rajasthan             38
Odisha                38
Gujarat               34
Punjab                32
Jharkhand             29
Assam                 22
Kerala                17
Uttarakhand           14
Jammu & Kashmir       12
Haryana               12
Chhattisgarh           8
Himachal Pradesh       6
Goa                    5
Nagaland               4
Tripura                4
Andaman & Nicobar      3
Mizoram                3
Meghalaya              3
Manipur                3
Sikkim                 2
Arunachal Pradesh      2
Daman and Diu          1
Chandigarh             1
Lakshadweep            1
Delhi                  1
Name: city, dtype: int64


In [10]:
import pandas as pd

df = pd.read_csv('/Users/krishnagawande/agro/tempdata/weather_clustered_cities.csv')

for state, group in df.groupby('state'):
    print(f"\n{state} ({len(group)} cities)")
    print(group['city'].sort_values().to_list())



Andaman & Nicobar (3 cities)
['Diglipur', 'Port Blair', 'Rongat']

Andhra Pradesh (152 cities)
['Advi Devalpalli', 'Alipur', 'Amalapuram', 'Arrapalli', 'Arutla', 'Ashwaraopeta', 'Avanigadda', 'Badurpalle', 'Badvel', 'Bayaram', 'Bayyanagudem', 'Bayyavaram', 'Betamcherla', 'Boddikurapadu', 'Bomareddipalli', 'Bommagondanahalli', 'Budili', 'Chicacole', 'Chinna Mushidivada', 'Chinna Orampadu', 'Chintakunta', 'Chintapalle', 'Chitrada', 'Chityal', 'Coringa', 'Damargidda', 'Dhanwada', 'Dilawarpur', 'Domkonda', 'Elurupadu', 'Enamadala', 'Enumulapalle', 'Ernagudem', 'Erraballa', 'Gadwal', 'Gajwel', 'Gandhari', 'Gandlapenta', 'Ganguru', 'Garsekurti', 'Giddalur', 'Golet', 'Gooty', 'Gudimulakhandrika', 'Gudipallipadu', 'Gudlavalleru', 'Gundugolanu', 'Hindupur', 'Holalagondi', 'Hyderabad', 'Ichora', 'Inole', 'Iskapalem', 'Itikalapalle', 'Jangaon', 'Kalikiri', 'Kaluvaya', 'Kalwakurti', 'Kambaduru', 'Kamepalle', 'Kanchika', 'Kandanati', 'Kandulapuram', 'Kanigiri', 'Kanteru', 'Karankot', 'Karlapalem',